# Modeling cyclobutadiene with the Extended Hubbard model and quantum phase estimation

## Background: cyclobutadiene and the automerization problem

Cyclobutadiene (C₄H₄) is the smallest cyclic molecule with 4 π-electrons, making it the prototypical **antiaromatic** system.
Unlike benzene (6 π-electrons, aromatic, stable), cyclobutadiene is highly reactive and has never been isolated at room temperature.

Hückel's rule predicts that cyclic molecules with 4*n* π-electrons (here *n* = 1) are destabilized by electron delocalization
rather than stabilized.
This is directly observable in cyclobutadiene's geometry: the molecule distorts from a symmetric square ($D_{4h}$) to a
rectangle ($D_{2h}$), alternating between two equivalent rectangular structures through a process called
**automerization**:

$$\text{rectangle}\ (D_{2h}) \;\rightleftharpoons\; \text{square}\ (D_{4h}) \;\rightleftharpoons\; \text{rectangle}\ (D_{2h})$$

The square geometry is the **transition state** of this reaction.
At this geometry, two frontier orbitals become degenerate (equal in energy), meaning a single Slater determinant (i.e., a mean-field approach like Hartree-Fock) cannot correctly describe the electronic structure.
This makes square cyclobutadiene a compact, well-understood test case for **strongly correlated** quantum chemistry methods — including quantum algorithms.

This notebook demonstrates a complete workflow from defining the chemical problem, through deriving a model Hamiltonian, to estimating the ground-state energy using iterative Quantum Phase Estimation (iQPE).
The companion notebook [`qpe_stretched_n2.ipynb`](qpe_stretched_n2.ipynb) walks through a similar workflow using the full *ab initio* Hamiltonian; this notebook focuses on the complementary approach of mapping the problem onto a simpler **model Hamiltonian**.

In addition to [installing `qdk-chemistry`](https://github.com/microsoft/qdk-chemistry/blob/main/INSTALL.md), you will need to install the `jupyter` extra to run this notebook:

```bash
pip install 'qdk-chemistry[jupyter]'
```

This installs the additional dependencies required by this notebook (ipykernel, pandas, and pyscf).

In [1]:
# Load frequently used external packages
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

# Reduce logging output for demo
from qdk_chemistry.utils import Logger
Logger.set_global_level(Logger.LogLevel.off)

## Why use a model Hamiltonian?

The companion notebook [`qpe_stretched_n2.ipynb`](qpe_stretched_n2.ipynb) shows how to run quantum phase estimation on the *full* molecular Hamiltonian derived from first-principles (*ab initio*) quantum chemistry.
That approach is general and accurate, but it comes at a cost: the number of qubits and quantum gates grows with the size of the molecular orbital basis set, which can make even modest molecules expensive to simulate.

An alternative is to **map the essential physics onto a simpler model Hamiltonian** that captures the key interactions with far fewer degrees of freedom.
For cyclobutadiene's automerization, the relevant physics lives in the 4 carbon π-orbitals. Rather than treating all electrons and orbitals, we can describe the system with a 4-site lattice model.

The [Extended Hubbard model](https://en.wikipedia.org/wiki/Hubbard_model) is a natural choice for conjugated π-systems.
It retains three physically motivated parameters:
- **$t$ (hopping)**: the energy for an electron to hop between neighboring sites (analogous to bond strength)
- **$U$ (on-site repulsion)**: the energy penalty when two electrons occupy the same site (governs correlation strength)
- **$V$ (nearest-neighbor repulsion)**: the Coulomb repulsion between electrons on adjacent sites (important in π-systems where charge fluctuations extend beyond a single atom)

This reduction from a full molecular Hamiltonian (many orbitals, many integrals) to 3 effective parameters has two practical benefits:

1. **Fewer qubits**: 4 sites × 2 spins = 8 qubits, compared to potentially dozens for the full orbital basis
2. **Physical insight**: the ratio $U/t$ directly tells you how strongly correlated the system is: large $U/t$ means electrons strongly avoid each other, which is exactly the regime where classical mean-field methods fail and quantum algorithms can provide an advantage

The key question is: *can we derive these parameters from first principles rather than fitting them by hand?*
The answer is yes, and `qdk-chemistry` provides the tools to do exactly that.

## Deriving the model from the molecule

The first step is to load the molecular structure.
Cyclobutadiene in its square ($D_{4h}$) geometry has all four C–C bonds equal in length — this is the transition state of the automerization reaction.
The 4 hydrogen atoms complete the valence of each carbon but play no role in the π-system.

We load the structure from an [XYZ-format](https://en.wikipedia.org/wiki/XYZ_file_format) file and visualize it to confirm the square geometry.

In [2]:
from qdk_chemistry.data import Structure
from qdk_chemistry.algorithms import create
from qdk.widgets import MoleculeViewer

# Load cyclobutadiene structure from XYZ file (D4h square transition state)
structure = Structure.from_xyz_file(Path("data/cyclobutadiene.structure.xyz"))

# Visualize the cyclobutadiene structure
MoleculeViewer(molecule_data=structure.to_xyz())

## Identifying the π-orbitals

In planar organic molecules like cyclobutadiene, the electron orbitals separate into two types:
- **σ-orbitals** lie in the molecular plane and form the strong C–C and C–H single bonds that define the molecular skeleton
- **π-orbitals** extend above and below the plane, formed by the overlap of carbon $p_z$ atomic orbitals (perpendicular to the ring)

The chemistry of interest (bond alternation, antiaromaticity, and strong correlation) lives entirely in the π-system.
Our goal is to isolate these 4 π-orbitals (one per carbon) from the full set of molecular orbitals, so we can build a 4-site model Hamiltonian.

First, we run a Hartree-Fock (HF) self-consistent field calculation to obtain a set of molecular orbitals.
We use the minimal STO-3G basis set, which is sufficient for extracting the model parameters.

In [3]:
# Run SCF
scf_solver = create("scf_solver")
E_hf, wfn_hf = scf_solver.run(structure, charge=0, spin_multiplicity=1, basis_or_guess="sto-3g")
print(f"Hartree-Fock energy: {E_hf:.6f} Hartree")

Hartree-Fock energy: -151.675049 Hartree


We can visualize the molecular orbitals near the Fermi level (HOMO-2 through LUMO+2) using the `MoleculeViewer` widget's data overlay feature, which displays the orbital energy and occupation alongside each isosurface.

In [4]:
from utils.viz_utils import generate_cube_data_with_info

# Identify orbitals from HOMO-2 to LUMO+2
hf_orbitals = wfn_hf.get_orbitals()
n_total_electrons = sum(wfn_hf.get_total_num_electrons())
n_occupied = n_total_electrons // 2
n_mo = hf_orbitals.get_num_molecular_orbitals()

homo_minus_2 = max(0, n_occupied - 3)
lumo_plus_2 = min(n_mo - 1, n_occupied + 2)
frontier_indices = list(range(homo_minus_2, lumo_plus_2 + 1))

# Generate cube data with energy/occupation overlays
cube_data = generate_cube_data_with_info(hf_orbitals, n_occupied, indices=frontier_indices)

MoleculeViewer(molecule_data=structure.to_xyz(), cube_data=cube_data)

To select the π-orbitals automatically, we use the Automated Valence Active Space (AVAS) method.
AVAS works by projecting the molecular orbitals onto a target set of atomic orbitals (here, the carbon $2p_z$ orbitals) and selecting those with the largest overlap.
This gives us a CAS(4,4) active space: **4 electrons** in **4 orbitals**, corresponding to the 4 π-electrons distributed across 4 carbon sites.

In [5]:
import qdk_chemistry.plugins.pyscf  # Enable PySCF plugin  # noqa: F401

# Use AVAS to select the π-orbitals by projecting onto the carbon 2pz atomic orbitals
avas_selector = create("active_space_selector", "pyscf_avas", ao_labels=["C 2pz"])
avas_wfn = avas_selector.run(wfn_hf)
indices, _ = avas_wfn.get_orbitals().get_active_space_indices()
n_active_e = sum(avas_wfn.get_active_num_electrons())
print(f"AVAS selected CAS({n_active_e},{len(indices)}) active space")
print(f"  Active orbital indices: {list(indices)}")

AVAS selected CAS(4,4) active space
  Active orbital indices: [12, 13, 14, 15]


We can verify the selection by visualizing the orbitals.
For a 4-site cyclic π-system, we expect the familiar nodal pattern from Hückel theory: one orbital with no nodes, two orbitals with one node each, and one orbital with two nodes.
In the square ($D_{4h}$) geometry, the two single-node orbitals are related by the $C_4$ rotational symmetry of the ring, making them symmetry-equivalent.
With 4 π-electrons to distribute, exactly 2 electrons must be shared between these two equivalent orbitals. No single assignment is preferred by symmetry, which is the root cause of the strong correlation in this system.

In [6]:
from qdk_chemistry.utils.cubegen import generate_cubefiles_from_orbitals

# Visualize the AVAS-selected active space orbitals (canonical)
active_orbitals = avas_wfn.get_orbitals()
cube_data = generate_cubefiles_from_orbitals(
    orbitals=active_orbitals,
    grid_size=(40, 40, 40),
    margin=10.0,
    indices=indices,
)

MoleculeViewer(molecule_data=structure.to_xyz(), cube_data=cube_data)

### Localizing the orbitals to define lattice sites

The canonical orbitals above are delocalized over the entire ring, which is the natural output of a Hartree-Fock calculation.
However, the Extended Hubbard model describes electrons hopping between **localized sites**, so we need one orbital per carbon atom.

We apply a Pipek-Mezey localization to transform the 4 delocalized π-orbitals into 4 site-localized orbitals, each centered on a single carbon.
These localized orbitals will serve as the "sites" in our model Hamiltonian.

In [7]:
from qdk_chemistry.utils.cubegen import generate_cubefiles_from_orbitals

# Localize the active-space orbitals for clearer visualization
localizer = create("orbital_localizer", "qdk_pipek_mezey")
loc_wfn = localizer.run(avas_wfn, *avas_wfn.get_orbitals().get_active_space_indices())

# Visualize the localized active space orbitals
loc_orbitals = loc_wfn.get_orbitals()
loc_indices, _ = loc_orbitals.get_active_space_indices()
cube_data = generate_cubefiles_from_orbitals(
    orbitals=loc_orbitals,
    grid_size=(40, 40, 40),
    margin=10.0,
    indices=loc_indices,
)

MoleculeViewer(molecule_data=structure.to_xyz(), cube_data=cube_data)

## Extracting Extended Hubbard parameters from the molecule

With localized π-orbitals in hand, we can now read off the Extended Hubbard parameters directly from the *ab initio* Hamiltonian integrals.

The full Extended Hubbard Hamiltonian is:

$$\hat{H}_\text{Ext. Hubbard} = \varepsilon \sum_i \hat{n}_i - t\sum_{\langle i,j \rangle} (\hat{a}_i^\dagger \hat{a}_j + \text{h.c.}) + U \sum_i \hat{n}_{i\uparrow} \hat{n}_{i\downarrow} + \frac{1}{2}\sum_{\langle i,j \rangle} V_{ij}\, (\hat{n}_i - z_i)(\hat{n}_j - z_j)$$

where $\hat{n}_i = \hat{n}_{i\uparrow} + \hat{n}_{i\downarrow}$ is the electron count on site $i$, and $z_i = 1$ is the effective nuclear charge (one π-electron per carbon). For a review of this model and the closely related Pariser-Parr-Pople (PPP) Hamiltonian, see [Fabian, Glaser and Solomon, *Digital Discovery*, 2026, **5**, 482–496](https://doi.org/10.1039/D5DD00445D).

We extract the parameters as follows:
- **$t$** (hopping): the largest off-diagonal element of the one-body integral matrix for each orbital identifies its nearest neighbor, the average of these gives $t$
- **$U$** (on-site repulsion): the average of the diagonal two-body integrals $\langle ii|ii \rangle$
- **$\varepsilon$** (on-site energy): set to zero, as it shifts all energies by a constant and does not affect the physics

Note that orbital localization does not preserve any particular ordering of the orbitals, so we identify nearest-neighbor connections by the magnitude of the hopping integrals rather than by matrix index.

In [8]:
n_sites = len(indices)
n_active_alpha, n_active_beta = avas_wfn.get_active_num_electrons()

# Construct the active-space Hamiltonian from the localized orbitals
hamiltonian_constructor = create("hamiltonian_constructor")
refined_orbitals = loc_wfn.get_orbitals()
active_hamiltonian = hamiltonian_constructor.run(refined_orbitals)

# Get one-body integrals of the active space
h1_alpha, _ = active_hamiltonian.get_one_body_integrals()
print(f"One-body integrals ({n_sites}x{n_sites}):\n{np.round(h1_alpha, 4)}")

# Hopping integral: identify nearest-neighbour hoppings from the largest
# off-diagonal elements in the one-body integral matrix.
nn_hoppings = []
for i in range(n_sites):
    row = np.abs(h1_alpha[i].copy())
    row[i] = 0.0  # exclude diagonal
    # The largest off-diagonal element is the nearest-neighbour hopping
    nn_hoppings.append(np.max(row))
t = float(np.mean(nn_hoppings))

# On-site Coulomb repulsion U: average of diagonal two-body integrals (i,i,i,i)
U_values = []
for i in range(n_sites):
    U_values.append(active_hamiltonian.get_two_body_element(i, i, i, i))
U = float(np.mean(U_values))

print(f"\nExtended Hubbard parameters from CAS({n_active_alpha + n_active_beta},{n_sites}) active space:")
print(f"  Hopping integral t = {t:.4f} Hartree")
print(f"  On-site repulsion U = {U:.4f} Hartree")

One-body integrals (4x4):
[[-1.2241 -0.0054 -0.1266 -0.1261]
 [-0.0054 -1.2242 -0.1261 -0.1265]
 [-0.1266 -0.1261 -1.224  -0.0052]
 [-0.1261 -0.1265 -0.0052 -1.224 ]]

Extended Hubbard parameters from CAS(4,4) active space:
  Hopping integral t = 0.1265 Hartree
  On-site repulsion U = 0.6898 Hartree


### Building the lattice Hamiltonian

Cyclobutadiene's π-system has a **ring topology**: each carbon is bonded to two neighbors, forming a cyclic 4-site chain.
We represent this as a periodic lattice and compute the remaining parameter, the nearest-neighbor repulsion $V$, using the **Ohno potential** that transitions smoothly between the on-site $U$ at zero distance and the classical $1/R$ Coulomb interaction at large separations.

The `create_ppp_hamiltonian` function assembles the full Extended Hubbard Hamiltonian from these ingredients.

In [9]:
from qdk_chemistry.data import LatticeGraph
from qdk_chemistry.utils.model_hamiltonians import create_ppp_hamiltonian, ohno_potential

# Build periodic chain lattice (ring topology)
lattice = LatticeGraph.chain(n=n_sites, periodic=True)

# Compute nearest-neighbour C-C bond length from the structure (coordinates are in Bohr)
coords = structure.get_coordinates()
carbon_coords = coords[:n_sites]  # first n_sites atoms are carbons
cc_distances = [np.linalg.norm(carbon_coords[i] - carbon_coords[(i + 1) % n_sites]) for i in range(n_sites)]
R_CC = float(np.mean(cc_distances))
print(f"Average C-C bond length: {R_CC:.4f} Bohr")

# Compute nearest-neighbour intersite Coulomb repulsion V via Ohno potential
V = ohno_potential(lattice, U=U, R=R_CC, nearest_neighbor_only=True)

# Build Extended Hubbard Hamiltonian (PPP with nearest-neighbour V only)
# epsilon=0 because it only shifts all energies by a constant (see discussion above)
hamiltonian = create_ppp_hamiltonian(lattice, epsilon=0.0, t=t, U=U, V=V, z=1.0)

# Extract integrals for display
h1 = hamiltonian.get_one_body_integrals()[0]  # one-body integral matrix
Vij = V[0, 1]

print(f"Extended Hubbard model: {lattice.num_sites} sites (ring), {n_active_alpha}α + {n_active_beta}β electrons")
print(f"One-body integrals:\n{np.round(h1, 4)}")
print(f"Two-body integrals (U): {U:.4f}")
print(f"Two-body integrals (V): {Vij:.4f}")
print(f"Ratio U/t = {U / t:.2f}, V/t = {Vij / t:.2f}")

Average C-C bond length: 2.7212 Bohr
Extended Hubbard model: 4 sites (ring), 2α + 2β electrons
One-body integrals:
[[-0.6487 -0.1265  0.     -0.1265]
 [-0.1265 -0.6487 -0.1265  0.    ]
 [ 0.     -0.1265 -0.6487 -0.1265]
 [-0.1265  0.     -0.1265 -0.6487]]
Two-body integrals (U): 0.6898
Two-body integrals (V): 0.3243
Ratio U/t = 5.45, V/t = 2.56


## Solving the model classically (reference energy)

Before running any quantum algorithm, we need a classical reference answer to benchmark against.
With only 4 sites, the Extended Hubbard model is small enough to solve exactly on a classical computer via full configuration interaction (here called CASCI).
This gives us the exact ground-state energy and wavefunction, which we'll compare to the QPE estimate later.

The configuration weights reveal the multi-reference character as expected for the degenerate square geometry, **no single electron configuration dominates** the ground state.

In [10]:
from qdk.widgets import Histogram

# Exact diagonalization (CASCI)
mc_calculator = create("multi_configuration_calculator", "macis_cas")
e_exact, wfn_exact = mc_calculator.run(hamiltonian, n_active_alpha, n_active_beta)
print(f"Exact ground state energy: {e_exact:.6f} Hartree")

# Plot top configuration weights from the CASCI wavefunction
num_configurations = len(wfn_exact.get_active_determinants())
print(f"Total configurations in the CASCI wavefunction: {num_configurations}")
top_dets = wfn_exact.get_top_determinants()
display(
    Histogram(
        bar_values={k.to_string(): np.abs(v)**2 for k, v in top_dets.items()},
        items="top-25",
        sort="high-to-low",
    )
)

Exact ground state energy: -0.304196 Hartree
Total configurations in the CASCI wavefunction: 36


## Preparing a trial state for quantum phase estimation

Quantum phase estimation requires an initial **trial state** that has significant overlap with the true ground state.
We construct a sparse trial state from the dominant configurations of the exact wavefunction — this is physically motivated by the determinant structure we observed above. For this particular problem, there are two configurations with large magnitude and four more with significant magnitude, leading to a total of 6 configurations which comprise the majority of overlap with the exact state.

For a detailed discussion of trial-state preparation and the trade-offs involved, see the companion notebook [`qpe_stretched_n2.ipynb`](qpe_stretched_n2.ipynb).

In [11]:
from qdk_chemistry.data import Wavefunction, CasWavefunctionContainer

# Use largest determinants from CASCI as trial states
NUM_TRIAL_DETS = 6

# Select top determinants and renormalize
dets = wfn_exact.get_top_determinants(NUM_TRIAL_DETS)
total_weight = sum(np.abs(c)**2 for c in dets.values())
dets = {det: c / np.sqrt(total_weight) for det, c in dets.items()}

# Build trial wavefunction
det_keys = list(dets.keys())
coeffs = np.array(list(dets.values()))
wfn_trial = Wavefunction(CasWavefunctionContainer(coeffs, det_keys, wfn_exact.get_orbitals()))

# Compute fidelity with exact wavefunction
exact_coeffs = np.array([wfn_exact.get_coefficient(det) for det in det_keys])
fidelity = np.abs(np.vdot(exact_coeffs, coeffs))**2
print(f"Overlap (fidelity) of trial state with CASCI wavefunction: {fidelity:.4f}")

# Plot trial state configuration weights
display(
    Histogram(
        bar_values={k.to_string(): np.abs(v)**2 for k, v in dets.items()},
        sort="high-to-low",
    )
)

Overlap (fidelity) of trial state with CASCI wavefunction: 0.6700


We load the trial state onto the quantum computer using the GF2+X sparse isometry method, which exploits the sparsity of chemistry wavefunctions to produce compact circuits.
See [`qpe_stretched_n2.ipynb`](qpe_stretched_n2.ipynb) for a comparison with other state preparation methods.

In [12]:
from qdk.widgets import Circuit

# Generate state preparation circuit for the sparse state via GF2+X sparse isometry
state_prep = create("state_prep", "sparse_isometry_gf2x")
state_prep_circuit = state_prep.run(wfn_trial)

# Visualize the sparse isometry circuit
display(Circuit(state_prep_circuit.get_qsharp_circuit()))

## Estimating the ground-state energy with quantum phase estimation

We now use iterative Quantum Phase Estimation (iQPE) to estimate the ground-state energy of the Extended Hubbard model on a quantum simulator.
For details on how iQPE works (ancilla qubit, controlled-$U^{2^k}$ applications, phase feedback), see [`qpe_stretched_n2.ipynb`](qpe_stretched_n2.ipynb).

The key advantage of running QPE on the model Hamiltonian rather than the full molecular Hamiltonian is **circuit depth**: with only 8 qubits (4 sites × 2 spins), the circuits are significantly shallower and more amenable to near-term fault-tolerant hardware.

First, we map the fermionic Hamiltonian to qubit operators using the Jordan-Wigner transformation.

In [13]:
# Prepare the qubit-mapped Hamiltonian
qubit_mapper = create("qubit_mapper", algorithm_name="qdk", encoding="jordan-wigner")
qubit_hamiltonian = qubit_mapper.run(hamiltonian)
print("Qubit Hamiltonian:\n", qubit_hamiltonian.get_summary())

Qubit Hamiltonian:
 Qubit Hamiltonian
  Number of qubits: 8
  Number of terms: 45
  Encoding: jordan-wigner
  Fermion mode order: blocked



In [14]:
# Set up parameters for iQPE
M_PRECISION = 8
SHOTS_PER_BIT = 3
SIMULATOR_SEED = 42

# Compute the evolution time from the Schatten norm bound: t = pi / ||H||
evolution_time = np.pi / qubit_hamiltonian.schatten_norm
print(f"Proposed evolution time: {evolution_time:.4f} Hartree^-1")

Proposed evolution time: 0.6300 Hartree^-1


Each iQPE iteration applies a controlled time-evolution operator to the system qubits, conditioned on the ancilla.
The time evolution is approximated using a 2nd-order Trotter-Suzuki decomposition.
Below we visualize one iteration of the circuit.

In [15]:
# Use factory methods to create the iQPE algorithm components
unitary_builder = create("unitary_builder", "trotter", order=2)
circuit_mapper = create("controlled_circuit_mapper", "pauli_sequence")
iqpe = create(
    "phase_estimation",
    "iterative",
    num_bits=M_PRECISION,
    evolution_time=evolution_time,
    shots_per_bit=SHOTS_PER_BIT,
)

# Generate the iQPE iteration circuit for a specific iteration (3rd from last)
iqpe_iter_circuit = iqpe.create_iteration_circuit(
    state_preparation=state_prep_circuit,
    qubit_hamiltonian=qubit_hamiltonian,
    unitary_builder=unitary_builder,
    circuit_mapper=circuit_mapper,
    iteration=M_PRECISION - 3,
    total_iterations=M_PRECISION,
)

# Visualize the iQPE iteration circuit
display(Circuit(iqpe_iter_circuit.get_qsharp_circuit()))

AttributeError: 'IterativePhaseEstimation' object has no attribute '_create_time_evolution'

### Single-trial run

We run a single iQPE trial on the `qdk` full-state simulator to get an initial energy estimate.

In [ ]:
# Execute the iQPE algorithm for a single trial
circuit_executor = create("circuit_executor", "qdk_full_state_simulator", seed=SIMULATOR_SEED)
result = iqpe.run(
    state_preparation=state_prep_circuit,
    qubit_hamiltonian=qubit_hamiltonian,
    circuit_executor=circuit_executor,
    unitary_builder=unitary_builder,
    circuit_mapper=circuit_mapper,
)

# Summarize the QPE results
estimated_energy = result.raw_energy + hamiltonian.get_core_energy()
estimated_error = abs(estimated_energy - e_exact)
print(f"QPE Results for {M_PRECISION}-bit precision:")
print(f"Reference CASCI energy: {e_exact:.6f} Hartree")
print(f"Total energy from phase estimation: {estimated_energy:.6f} Hartree")
print(f"Energy difference with CASCI energy: {estimated_error:.3e} Hartree")

### Multi-trial run for statistical confidence

A single iQPE trial can return incorrect phase bits due to the probabilistic nature of quantum measurement, particularly when the trial state has imperfect overlap with the ground state.
Running multiple trials and taking the **majority-vote** energy (most frequently observed value) provides a robust estimate.

In [ ]:
NUM_TRIALS = 20
RESULTS_DIR = Path(
    f"results_iqpe_cyclobutadiene/precision_{M_PRECISION}/time_{round(evolution_time, 12)}"
)

# Run iQPE if results do not already exist
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
for trial in range(NUM_TRIALS):
    trial_seed = SIMULATOR_SEED + trial
    json_path = RESULTS_DIR / f"iqpe_result_{trial_seed}.qpe_result.json"
    if not json_path.exists():
        print(f"Running trial {trial + 1} of {NUM_TRIALS}...")
        executor = create(
            "circuit_executor",
            "qdk_full_state_simulator",
            type="cpu",
            seed=trial_seed,
        )
        result = iqpe.run(
            state_preparation=state_prep_circuit,
            qubit_hamiltonian=qubit_hamiltonian,
            circuit_executor=executor,
            unitary_builder=unitary_builder,
            circuit_mapper=circuit_mapper,
        )
        result.to_json_file(json_path)

We collect the results from all trials and identify the majority-vote energy estimate.

In [ ]:
from qdk_chemistry.data import QpeResult

# Load the results
results_loaded = []
for json_file in RESULTS_DIR.glob("*qpe_result.json"):
    result = QpeResult.from_json_file(json_file)
    results_loaded.append(result)

# Count the energy values
energy_counts = Counter(
    [
        result.raw_energy + hamiltonian.get_core_energy()
        for result in results_loaded
    ]
)
print(f"QPE Results of {M_PRECISION} bit precision from {NUM_TRIALS} trials:")
display(pd.DataFrame(energy_counts.items(), columns=['Energy (Hartree)', 'Counts']))

# Use the most frequently observed energy across all trials as the overall estimate
estimated_energy, _ = energy_counts.most_common(1)[0]

Finally, we compare the QPE estimate to the exact classical result and visualize the distribution of energy errors across trials.

In [ ]:
# Print summary of results
print(f"Reference CASCI energy: {e_exact:.6f} Hartree")
print(f"Total energy from phase estimation: {estimated_energy:.6f} Hartree")
print(f"Energy difference with CASCI energy: {abs(estimated_energy - e_exact):.3e} Hartree")

# Summarize the energy errors
energy_errors = {
    qpe_e - e_exact: count
    for qpe_e, count in sorted(energy_counts.items())
}

# Plot distribution of energy differences
print("Energy difference (Hartree) distribution:")
display(
    Histogram(
        bar_values={f"{err:.3e}": count for err, count in energy_errors.items()}
    )
)